In [1]:
# Import libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

import joblib

print("Libraries imported successfully. ✅")

Libraries imported successfully. ✅


Load feature-engineered data

Why: Use the dataset produced by 03_feature_engineering.ipynb

In [2]:
# Load feature-engineered data

df = pd.read_csv(
    "../data/processed/feature_engineered_data.csv"
)

print("Dataset loaded successfully. ✅")
print("Shape:", df.shape)

Dataset loaded successfully. ✅
Shape: (112, 18)


Define features and target

Why: X contains inputs and y contains the prediction target.

In [3]:
# Define the target column

target = "is_unicorn"

# Define the feature columns

feature_columns = [
    "founded_year",
    "sector",
    "subsector",
    "country",
    "city",
    "funding_usd_millions",
    "valuation_usd_millions",
    "employees",
    "startup_stage",
    "startup_age",
    "funding_log",
    "valuation_log",
    "funding_per_employee"
]

# Create features and target

X = df[feature_columns].copy()
y = df[target].astype(int)

print("Features:", X.shape)
print("Target:", y.shape)

Features: (112, 13)
Target: (112,)


Identify numeric and categorical features

Why: Numerical and text columns need different preprocessing.

In [4]:
# Define numerical features

numeric_features = [
    "founded_year",
    "funding_usd_millions",
    "valuation_usd_millions",
    "employees",
    "startup_age",
    "funding_log",
    "valuation_log",
    "funding_per_employee"
]

# Define categorical features

categorical_features = [
    "sector",
    "subsector",
    "country",
    "city",
    "startup_stage"
]

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 8
Categorical features: 5


Split the data

Why: Keep some data unseen during training so we can evaluate the model fairly.

Because we only have 11 positive records, stratify=y is important.

In [5]:
# Split the data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (89, 13)
Testing data: (23, 13)


Create preprocessing pipeline

Why: Fill missing numeric values, scale numbers, and convert categorical values into numerical features.

In [6]:
# Preprocess numerical features

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Preprocess categorical features

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

# Combine both preprocessing pipelines

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Preprocessing pipeline created. ✅")

Preprocessing pipeline created. ✅


Model 1 — Logistic Regression

Why: A simple and interpretable baseline classification model.

In [7]:
# Create Logistic Regression pipeline

logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

# Train Logistic Regression

logistic_model.fit(
    X_train,
    y_train
)

print("Logistic Regression trained. ✅")

Logistic Regression trained. ✅


Model 2 — Random Forest

Why: Random Forest can capture nonlinear relationships between startup features.

In [8]:
# Create Random Forest pipeline

random_forest_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced"
    ))
])

# Train Random Forest

random_forest_model.fit(
    X_train,
    y_train
)

print("Random Forest trained. ✅")

Random Forest trained. ✅


Model 3 — Gradient Boosting

Why: Gradient Boosting is another strong model for structured/tabular data.

In [9]:
# Create Gradient Boosting pipeline

gradient_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingClassifier(
        n_estimators=100,
        random_state=42
    ))
])

# Train Gradient Boosting

gradient_model.fit(
    X_train,
    y_train
)

print("Gradient Boosting trained. ✅")

Gradient Boosting trained. ✅


Evaluate models

Why: Compare models using metrics that matter for an imbalanced classification problem.

In [10]:
# Create a function to evaluate a model

def evaluate_model(model, name):

    predictions = model.predict(X_test)

    probabilities = model.predict_proba(X_test)[:, 1]

    print("\n", name)

    print(
        "Accuracy:",
        round(accuracy_score(y_test, predictions), 3)
    )

    print(
        "Precision:",
        round(precision_score(
            y_test,
            predictions,
            zero_division=0
        ), 3)
    )

    print(
        "Recall:",
        round(recall_score(
            y_test,
            predictions,
            zero_division=0
        ), 3)
    )

    print(
        "F1 Score:",
        round(f1_score(
            y_test,
            predictions,
            zero_division=0
        ), 3)
    )

    print(
        "ROC-AUC:",
        round(roc_auc_score(
            y_test,
            probabilities
        ), 3)
    )

Evaluate Logistic Regression

In [11]:
# Evaluate Logistic Regression

evaluate_model(
    logistic_model,
    "Logistic Regression"
)


 Logistic Regression
Accuracy: 0.87
Precision: 0.4
Recall: 1.0
F1 Score: 0.571
ROC-AUC: 0.976


Evaluate Random Forest

In [12]:
# Evaluate Random Forest

evaluate_model(
    random_forest_model,
    "Random Forest"
)


 Random Forest
Accuracy: 0.913
Precision: 0.5
Recall: 1.0
F1 Score: 0.667
ROC-AUC: 0.952


Evaluate Gradient Boosting

In [13]:
# Evaluate Gradient Boosting

evaluate_model(
    gradient_model,
    "Gradient Boosting"
)


 Gradient Boosting
Accuracy: 0.826
Precision: 0.25
Recall: 0.5
F1 Score: 0.333
ROC-AUC: 0.929


Compare models

Why: Put the results into a table so we can select the best model objectively.

In [14]:
# Store model results

models = {
    "Logistic Regression": logistic_model,
    "Random Forest": random_forest_model,
    "Gradient Boosting": gradient_model
}

results = []

for name, model in models.items():

    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(
            y_test,
            predictions
        ),
        "Precision": precision_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "F1": f1_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "ROC_AUC": roc_auc_score(
            y_test,
            probabilities
        )
    })

results_df = pd.DataFrame(results)

display(
    results_df.round(3)
)

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Logistic Regression,0.870,0.40,1.0,0.571,0.976
1,Random Forest,0.913,0.50,1.0,0.667,0.952
2,Gradient Boosting,0.826,0.25,0.5,0.333,0.929


Select the best model

Why: We'll select the model using F1 score rather than accuracy because our target is imbalanced.

In [15]:
# Select the model with the highest F1 score

best_model_name = (
    results_df
    .sort_values("F1", ascending=False)
    .iloc[0]["Model"]
)

best_model = models[best_model_name]

print("Best model:", best_model_name)

Best model: Random Forest


Save the best model

Why: The saved model will later be used by our Streamlit application to predict startup success.

In [16]:
# Save the best trained model

joblib.dump(
    best_model,
    "../models/startup_success_model.pkl"
)

print("Best model saved successfully. ✅")

Best model saved successfully. ✅


Save model results

Why: Keep the comparison results for the final report and dashboard.

In [17]:
# Save model comparison results

results_df.to_csv(
    "../reports/model_comparison.csv",
    index=False
)

print("Model results saved successfully. ✅")

Model results saved successfully. ✅


Final validation

Why: Confirm that the training stage completed successfully.

In [18]:
# Final model training validation

print("========== MODEL TRAINING COMPLETE ==========")

print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Features:", len(feature_columns))
print("Best model:", best_model_name)

print("\nModel saved:")
print("../models/startup_success_model.pkl")

========== MODEL TRAINING COMPLETE ==========
Training records: 89
Testing records: 23
Features: 13
Best model: Random Forest

Model saved:
../models/startup_success_model.pkl
